In [ ]:
'''
#data check
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockU_info — 读取 EXTR U.zm + merged U，检查结构并做基础处理

功能：
  1) 读取 EXTR:
       /mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/
       CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_U.zm.nc
     - 打印数据集结构、时间/纬度/垂直信息
     - 计算 60–90°N 极帽平均 U(time, lev) 的基本统计
  2) 读取 merged:
       /mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/
       BWCN.e122.f19_g16.merged.nc
     - 同样打印 U 变量结构与极帽平均结果
"""

import os
import numpy as np
import xarray as xr

# === 路径 ===
EXTR_U_PATH = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_U.zm.nc"
)
MERGED_PATH = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.merged.nc"
)


def guess_p_unit(lev_vals):
    """
    根据 lev 数值猜测单位：
      - 若 max(lev) <= 2000 → 认为是 hPa
      - 否则认为是 Pa
    返回 (lev_hpa, unit_str)，其中 lev_hpa 是以 hPa 表示的数值
    """
    lev_vals = np.asarray(lev_vals)
    max_lev = np.nanmax(lev_vals)
    if max_lev <= 2000:  # already hPa
        return lev_vals, "hPa"
    else:  # assume Pa
        return lev_vals / 100.0, "Pa(→hPa)"


def find_u_var(ds):
    """在 ds 中找到 U 变量名。优先使用 'U'，否则找名字里含 'U' 的变量."""
    if "U" in ds.data_vars:
        return "U"
    candidates = [v for v in ds.data_vars if "U" in v.upper()]
    if not candidates:
        raise KeyError(f"No U-like variable found, data_vars={list(ds.data_vars)}")
    return candidates[0]


def compute_polar_cap_u(ds, u_name="U"):
    """
    从 ds 中取 U，做：
      - 如有 lon 先经向平均
      - 60–90N 余弦加权极帽平均
    返回:
      U_pcap: (time, lev_dim) DataArray
      lev_name: 垂直维名称
      lev_vals: 原始垂直坐标值
      lev_hpa: 以 hPa 表示的垂直坐标值
      lev_unit: 推断的单位说明字符串
    """
    U = ds[u_name]

    # 先 zonal mean（EXTR 的 .zm 文件可能本来就没有 lon）
    if "lon" in U.dims:
        U = U.mean(dim="lon")

    # 纬向极帽平均
    if "lat" not in U.dims:
        raise ValueError(f"U has no 'lat' dimension, dims={U.dims}")
    U_cap = U.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(U_cap["lat"]))
    U_pcap = U_cap.weighted(weights).mean(dim="lat")  # (time, lev_dim)

    # 找垂直坐标名（注意 EXTR U.zm 用的是 lev_p）
    lev_name = None
    for name in ("plev", "lev", "level", "lev_p"):
        if name in U_pcap.dims:
            lev_name = name
            break
    if lev_name is None:
        raise ValueError(
            f"U_pcap has no vertical dim among ('plev','lev','level','lev_p'), "
            f"dims={U_pcap.dims}"
        )

    lev_vals = U_pcap[lev_name].values
    lev_hpa, lev_unit = guess_p_unit(lev_vals)

    return U_pcap, lev_name, lev_vals, lev_hpa, lev_unit


def print_time_info(da, label="[time]"):
    """打印时间维度信息（假定存在 time）"""
    time = da["time"]
    print(f"\n{label} Time info:")
    try:
        years = time.dt.year.values.astype(int)
        doy = time.dt.dayofyear.values.astype(int)
        years_u = np.unique(years)
        n_days = int(doy.max())
        print("  time[0] :", str(time.values[0]))
        print("  time[-1]:", str(time.values[-1]))
        print("  years range:", years_u[0], "→", years_u[-1])
        print("  n_years   :", len(years_u))
        print("  max dayofyear (n_days):", n_days)
    except Exception as e:
        print("  WARNING: failed to decode .dt on time:", repr(e))
        print("  Raw time values example:", time.values[:5])


def print_lat_info(da, label="[lat]"):
    if "lat" not in da.dims:
        print(f"\n{label} WARNING: no 'lat' dimension, dims=", da.dims)
        return
    lat = da["lat"].values
    print(f"\n{label} Latitude info:")
    print("  lat shape:", lat.shape)
    print("  lat min / max:", float(np.nanmin(lat)), "/", float(np.nanmax(lat)))


def inspect_dataset_u(path, tag):
    """
    通用检查函数：
      - 打开 path
      - 找 U 变量
      - 打印结构、时间/纬度/垂直信息
      - 计算极帽 U_pcap(time, lev) 及其统计量
      - 特别打印 ~10 hPa 层的统计
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"{tag} file not found: {path}")

    print(f"\n========== [{tag}] Open dataset ==========")
    print("Path:", path)

    ds = xr.open_dataset(path)
    print(f"\n[{tag}] Dataset summary:\n")
    print(ds)

    # --- 找 U 变量 ---
    u_name = find_u_var(ds)
    U = ds[u_name]

    print(f"\n[{tag}] Using U variable name:", u_name)
    print(f"[{tag}] U dims:", U.dims)
    print(f"[{tag}] U shape:", U.shape)
    print(f"[{tag}] U attrs:", U.attrs)

    # --- 时间信息 ---
    if "time" not in U.dims:
        raise ValueError(f"[{tag}] U has no 'time' dimension!")
    print_time_info(U, label=f"[{tag}]")

    # --- 纬度信息 ---
    print_lat_info(U, label=f"[{tag}]")

    # --- 极帽平均 U(time, lev) ---
    print(f"\n[{tag}] Computing polar-cap (60–90N) zonal-mean U ...")
    U_pcap, lev_name, lev_vals, lev_hpa, lev_unit = compute_polar_cap_u(ds, u_name=u_name)

    print(f"[{tag}] U_pcap dims:", U_pcap.dims)
    print(f"[{tag}] U_pcap shape:", U_pcap.shape)
    print(f"[{tag}] U_pcap attrs:", U_pcap.attrs)

    print(f"\n[{tag}] Vertical coordinate info:")
    print("  lev name:", lev_name)
    print("  lev shape:", lev_vals.shape)
    print("  lev (first 10 raw):", lev_vals[:10])
    print("  lev (hPa, first 10):", lev_hpa[:10])
    print("  lev min / max (hPa):",
          float(np.nanmin(lev_hpa)), "/", float(np.nanmax(lev_hpa)))
    print("  guessed unit:", lev_unit)

    # --- 极帽 U 整体统计 ---
    u_mean = float(U_pcap.mean().values)
    u_std = float(U_pcap.std().values)
    print(f"\n[{tag}] U_pcap overall stats:")
    print("  mean:", u_mean)
    print("  std :", u_std)

    # --- ~10 hPa 层时间序列 ---
    try:
        target_lev = 10.0  # hPa
        idx_10 = int(np.argmin(np.abs(lev_hpa - target_lev)))
        lev_at = float(lev_hpa[idx_10])
        print(f"\n[{tag}] Nearest level to {target_lev} hPa: "
              f"{lev_at:.2f} hPa (index {idx_10})")

        # 把垂直维统一叫 lev 方便 isel
        if lev_name != "lev":
            U_pcap_named = U_pcap.rename({lev_name: "lev"})
        else:
            U_pcap_named = U_pcap

        u_10 = U_pcap_named.isel(lev=idx_10)  # (time,)
        print(f"[{tag}] U_pcap at ~10 hPa stats:")
        print("  mean:", float(u_10.mean().values))
        print("  std :", float(u_10.std().values))
        print("  sample (first 10 values):")
        print("   ", u_10.values[:10])
    except Exception as e:
        print(f"\n[{tag}] WARNING: failed to extract ~10 hPa time series:", repr(e))

    ds.close()
    print(f"\n[{tag}] Done.")


def main():
    # 1) EXTR U.zm
    inspect_dataset_u(EXTR_U_PATH, tag="EXTR_U.zm")

    # 2) merged U
    inspect_dataset_u(MERGED_PATH, tag="MERGED")


if __name__ == "__main__":
    main()
'''

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockA (U) — 读取 U 数据 & 预处理 & 写缓存
#
# 功能：
#   1) 从 EXTR U.zm 文件中，提取 60°N 经向平均风场 U(time, lev)
#   2) 从 merged 文件中，提取 60°N 经向平均风场 U(time, lev)
#   3) 把两个 time–height 序列写成 NetCDF，供 BlockB / BlockC / BlockD 使用
#   4) O3 极端年信息直接从 General_O3 目录中读取（不再重新计算）
#
# 路径：
#   O3_ROOT = /home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3
#   U_ROOT  = /home/weiji/test_code/plots/New_weiji_general_why0008important/General_U
#
#   EXTR U 输入：
#     /mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_U.zm.nc
#   merged U 输入：
#     /mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc
#
#   输出：
#     U_EXTR_60N_lev_time.nc   — U_60N(time, lev) [Pa]
#     U_MERGED_60N_lev_time.nc — U_60N(time, lev) [Pa]
# ============================================================

import os
import numpy as np
import xarray as xr

# 路径
O3_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
U_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_U"
os.makedirs(U_ROOT, exist_ok=True)

EXTR_U_PATH = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_U.zm.nc"
)
MERGED_PATH = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.merged.nc"
)

# O3 极端年 txt（已经由 O3 BlockA 生成）
LOW10_TXT = os.path.join(O3_ROOT, "O3_lowest10_years.txt")
LOW25_TXT = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

U_EXTR_60N_NC   = os.path.join(U_ROOT, "U_EXTR_60N_lev_time.nc")
U_MERGED_60N_NC = os.path.join(U_ROOT, "U_MERGED_60N_lev_time.nc")


def calc_u_60N_lev(ds, var="U"):
    """
    计算 60°N 的经向平均 U(time, lev)，并把垂直坐标统一为:
      - 维度名: 'lev'
      - 单位: Pa

    步骤：
      - 若有 lon → 先经向平均
      - 找到离 60°N 最近的格点
      - 识别垂直维 (plev, lev_p, lev, level)
      - 若最大 lev 值 <= 2000 → 认为是 hPa，乘 100 变 Pa
        否则认为已经是 Pa
    """
    da = ds[var]

    # zonal mean（EXTR U.zm 本身已经是 lon:mean，不过这里兜底）
    if "lon" in da.dims:
        da = da.mean(dim="lon")

    # 选 60°N（最近网格）
    if "lat" not in da.dims:
        raise ValueError(f"{var} has no 'lat' dimension, dims={da.dims}")
    lat_vals = da["lat"].values
    idx_lat = int(np.argmin(np.abs(lat_vals - 60.0)))
    da_60 = da.isel(lat=idx_lat)

    # 识别垂直维
    lev_name = None
    for name in ("plev", "lev_p", "lev", "level"):
        if name in da_60.dims:
            lev_name = name
            break
    if lev_name is None:
        raise ValueError(
            f"{var} at 60N has no vertical dim among ('plev','lev_p','lev','level'), "
            f"dims={da_60.dims}"
        )

    lev_vals = da_60[lev_name].values
    max_lev = float(np.nanmax(lev_vals))

    # 统一转成 Pa，维度名改为 'lev'
    if max_lev <= 2000.0:
        # 认为是 hPa → Pa
        lev_pa = lev_vals * 100.0
    else:
        # 已经是 Pa
        lev_pa = lev_vals

    da_60 = da_60.rename({lev_name: "lev"})
    da_60 = da_60.assign_coords(lev=("lev", lev_pa))

    return da_60  # (time, lev)，lev 单位 Pa


def main_blockA_U():
    # ======= 读 O3 极端年列表（仅检查是否存在） =======
    if not (os.path.exists(LOW10_TXT) and os.path.exists(LOW25_TXT)):
        raise FileNotFoundError(
            "O3 extreme-year txt files not found. Please run O3 BlockA first.\n"
            f"Expected: {LOW10_TXT} and {LOW25_TXT}"
        )

    lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockA_U] Lowest-10 O3 years (for reference):", lowest10_years)
    print("[BlockA_U] Lowest-25% O3 years (for reference):", lowest25_years)

    # ======= 1. EXTR U.zm → U_60N(time, lev) =======
    print("[BlockA_U] Reading EXTR U.zm:", EXTR_U_PATH)
    ds_extr = xr.open_dataset(EXTR_U_PATH)
    u_extr_60 = calc_u_60N_lev(ds_extr, var="U")  # (time, lev) Pa
    ds_extr.close()

    # 简单时间信息
    years_extr = np.unique(u_extr_60.time.dt.year.values.astype(int))
    n_days = int(u_extr_60.time.dt.dayofyear.max())
    print("[BlockA_U] EXTR U_60N years:", years_extr[0], "→", years_extr[-1])
    print("[BlockA_U] EXTR U_60N n_years:", len(years_extr), "n_days:", n_days)

    # 输出为 DataArray，名字 U_60N
    u_extr_60 = u_extr_60.transpose("time", "lev")
    u_extr_60.name = "U_60N"
    u_extr_60.attrs["long_name"] = "Zonal mean wind at 60N"
    u_extr_60.attrs["units"] = "m s-1"
    u_extr_60["lev"].attrs["units"] = "Pa"

    u_extr_60.to_netcdf(U_EXTR_60N_NC)
    print("[BlockA_U] Saved EXTR U_60N(time,lev) to:", U_EXTR_60N_NC)

    # ======= 2. merged U → U_60N(time, lev) =======
    print("[BlockA_U] Reading merged file:", MERGED_PATH)
    ds_merged = xr.open_dataset(MERGED_PATH)
    u_merged_60 = calc_u_60N_lev(ds_merged, var="U")  # (time, lev) Pa
    ds_merged.close()

    years_merged = np.unique(u_merged_60.time.dt.year.values.astype(int))
    n_days_m = int(u_merged_60.time.dt.dayofyear.max())
    print("[BlockA_U] merged U_60N years:", years_merged[0], "→", years_merged[-1])
    print("[BlockA_U] merged U_60N n_years:", len(years_merged), "n_days:", n_days_m)

    u_merged_60 = u_merged_60.transpose("time", "lev")
    u_merged_60.name = "U_60N"
    u_merged_60.attrs["long_name"] = "Zonal mean wind at 60N"
    u_merged_60.attrs["units"] = "m s-1"
    u_merged_60["lev"].attrs["units"] = "Pa"

    u_merged_60.to_netcdf(U_MERGED_60N_NC)
    print("[BlockA_U] Saved merged U_60N(time,lev) to:", U_MERGED_60N_NC)

    print("[BlockA_U] Done.")


if __name__ == "__main__":
    main_blockA_U()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB (U) — 60°N, 10 & 50 hPa zonal-mean wind 折线图
#
# 功能：
#   使用 BlockA_U 输出的 U_60N(time,lev)，以及 O3 定义的极端年列表，
#   绘制四张折线图：
#
#   1) EXTR：10 个极端低 O3 年 vs 其它年份（60°N, 10 hPa）
#   2) EXTR：10 个极端低 O3 年 vs 其它年份（60°N, 50 hPa）
#   3) merged：0008/0013/0014/0019 vs EXTR all / low25 气候态（60°N, 10 hPa）
#   4) merged：0008/0013/0014/0019 vs EXTR all / low25 气候态（60°N, 50 hPa）
#
#   时间坐标：与 O3 BlockB 一致：
#      x = 0..364, 0–90 = 前一年 Oct–Dec, 91–364 = 当年 Jan–Sep
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

O3_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
U_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_U"
os.makedirs(U_ROOT, exist_ok=True)

U_EXTR_60N_NC   = os.path.join(U_ROOT, "U_EXTR_60N_lev_time.nc")
U_MERGED_60N_NC = os.path.join(U_ROOT, "U_MERGED_60N_lev_time.nc")
LOW10_TXT       = os.path.join(O3_ROOT, "O3_lowest10_years.txt")
LOW25_TXT       = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

# 前一年拼接的天数（Oct–Dec）
N_PREV = 91

# merged 中关注的特殊年份
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)


def get_level_index(lev_vals_pa, target_hpa):
    """在 lev (Pa) 中找到最接近 target_hpa 的层索引"""
    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(lev_vals_pa - target_pa)))
    return idx, float(lev_vals_pa[idx] / 100.0)


def plot_extremes_vs_others(var_extr, lowest10_years, level_label, out_png, out_pdf):
    """
    画图：10 个极端低 O3 年 vs 其它年份气候态（EXTR）
    var_extr: DataArray(time,) 对应 60°N 某一层的 U
    """
    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))
    n_days = int(var_extr.time.dt.dayofyear.max())
    print(f"[BlockB_U] EXTR years for {level_label}:", years_extr)
    print(f"[BlockB_U] Days per year (EXTR): {n_days}")

    fig, ax = plt.subplots(1, 1, figsize=(13, 10), constrained_layout=True)

    low_years = lowest10_years
    neutral_years = years_extr[~np.isin(years_extr, low_years)]

    # 切分当前年 / 前一年
    var_low_cur = var_extr.sel(time=var_extr.time.dt.year.isin(low_years))
    var_low_prev = var_extr.sel(time=var_extr.time.dt.year.isin(low_years - 1))

    var_neu_cur = var_extr.sel(time=var_extr.time.dt.year.isin(neutral_years))
    var_neu_prev = var_extr.sel(time=var_extr.time.dt.year.isin(neutral_years - 1))

    # 非极端年日气候态
    neu_cur_mean = var_neu_cur.groupby("time.dayofyear").mean("time")
    neu_cur_std  = var_neu_cur.groupby("time.dayofyear").std("time")

    neu_prev_mean = var_neu_prev.groupby("time.dayofyear").mean("time")
    neu_prev_std  = var_neu_prev.groupby("time.dayofyear").std("time")

    neu_cur_mean = np.array(neu_cur_mean)
    neu_cur_std  = np.array(neu_cur_std)
    neu_prev_mean = np.array(neu_prev_mean)
    neu_prev_std  = np.array(neu_prev_std)

    # 极端年 reshape: (n_extreme, 365)
    n_extreme = len(low_years)
    var_low_cur_arr  = np.reshape(np.array(var_low_cur), (n_extreme, n_days))
    var_low_prev_arr = np.reshape(np.array(var_low_prev), (n_extreme, n_days))

    colors_ext = cm.twilight(np.linspace(0, 1, n_extreme))

    # extreme 年：前一年 Oct–Dec + 当年 Jan–Sep
    for j in range(n_extreme):
        # 当年 Jan–Sep: x = 91..364
        ax.plot(
            range(N_PREV, n_days),
            var_low_cur_arr[j, 0:n_days - N_PREV],
            color=colors_ext[j],
            alpha=0.8,
            linewidth=2,
            label="low O3 years" if j == 0 else None,
        )
        # 前一年 Oct–Dec: x = 0..90
        ax.plot(
            range(0, N_PREV),
            var_low_prev_arr[j, n_days - N_PREV:n_days],
            color=colors_ext[j],
            alpha=0.8,
            linewidth=2,
        )

    # 非极端年气候态 ±std
    ax.errorbar(
        range(N_PREV, n_days),
        neu_cur_mean[0:n_days - N_PREV],
        neu_cur_std[0:n_days - N_PREV],
        color="grey",
        alpha=0.5,
        elinewidth=1.5,
        linewidth=3,
        label="all other years",
    )
    ax.errorbar(
        range(0, N_PREV),
        neu_prev_mean[n_days - N_PREV:n_days],
        neu_prev_std[n_days - N_PREV:n_days],
        color="grey",
        alpha=0.5,
        elinewidth=1.5,
        linewidth=3,
    )

    ax.set_xlim(0, n_days)
    ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(
        ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
         "May", "June", "July", "Aug", "Sep"],
        fontsize=15,
    )

    ax.set_ylabel(f"U at 60°N, {level_label} (m s$^{{-1}}$)", fontsize=18)
    ax.set_yticklabels(ax.get_yticks(), size=18)
    ax.legend(fontsize=14)
    ax.set_title(f"Zonal mean wind at 60°N, {level_label} — low O$_3$ years vs others")

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)

    print("[BlockB_U] Saved extreme vs other years figure:")
    print("   ", out_png)
    print("   ", out_pdf)


def plot_special_vs_climatology(
    var_extr,
    var_merged,
    lowest25_years,
    level_label,
    out_png,
    out_pdf,
):
    """
    画图：merged 0008/13/14/19 vs EXTR 全体 & 低25% daily 气候态
    var_extr : EXTR U_60N at某层 (time,)
    var_merged: merged U_60N at某层 (time,)
    """
    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))
    n_days = int(var_extr.time.dt.dayofyear.max())
    print(f"[BlockB_U] EXTR years ({level_label}):", years_extr)
    print(f"[BlockB_U] Days per year:", n_days)

    # === EXTR all-years daily climatology（保持原样） ===
    clim_all_mean = var_extr.groupby("time.dayofyear").mean("time")
    clim_all_std  = var_extr.groupby("time.dayofyear").std("time")

    # === [FIX] EXTR low-25% daily climatology：用 Y-1 Oct–Dec + Y Jan–Sep ===
    #
    # 思路和 O3 一致：
    #   base_low25_cur  : year ∈ lowest25_years 的当年（用于 Jan–Sep）
    #   base_low25_prev : year ∈ lowest25_years-1 的前一年（用于 Oct–Dec）
    #   先分别按 dayofyear 做 climatology，再构造一个完整的
    #   clim_low25_full(dayofyear)：
    #       DOY 1..(n_days-N_PREV)：用 "当年" climatology
    #       DOY (n_days-N_PREV+1)..n_days：用 "前一年" climatology
    #   后面做 Oct–Dec/Jan–Sep 的拼接，都从这个 full climatology 上取。
    # --------------------------------------------------------------
    base_low25_cur = var_extr.sel(time=var_extr.time.dt.year.isin(lowest25_years))
    base_low25_prev = var_extr.sel(time=var_extr.time.dt.year.isin(lowest25_years - 1))

    clim_low25_cur_mean = base_low25_cur.groupby("time.dayofyear").mean("time")
    clim_low25_cur_std  = base_low25_cur.groupby("time.dayofyear").std("time")

    clim_low25_prev_mean = base_low25_prev.groupby("time.dayofyear").mean("time")
    clim_low25_prev_std  = base_low25_prev.groupby("time.dayofyear").std("time")

    # 从 “当年” 的 daily 气候态拷贝一份
    clim_low25_full_mean = clim_low25_cur_mean.copy()
    clim_low25_full_std  = clim_low25_cur_std.copy()

    # Oct–Dec 对应的 DOY（例如 noleap 时 275..365）
    oct_dec_doys = np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int)

    # 用前一年 climatology 替换 Oct–Dec 段
    clim_low25_full_mean.loc[dict(dayofyear=oct_dec_doys)] = (
        clim_low25_prev_mean.sel(dayofyear=oct_dec_doys).values
    )
    clim_low25_full_std.loc[dict(dayofyear=oct_dec_doys)] = (
        clim_low25_prev_std.sel(dayofyear=oct_dec_doys).values
    )

    # 转 numpy，后面再做 composite
    clim_all_mean = np.array(clim_all_mean)
    clim_all_std  = np.array(clim_all_std)
    clim_low25_mean = np.array(clim_low25_full_mean)
    clim_low25_std  = np.array(clim_low25_full_std)

    # === 把 EXTR 日气候态拼成 “前一年 Oct–Dec + 当年 Jan–Sep” ===
    all_prev_mean = clim_all_mean[n_days - N_PREV:n_days]
    all_prev_std  = clim_all_std[n_days - N_PREV:n_days]
    all_cur_mean  = clim_all_mean[0:n_days - N_PREV]
    all_cur_std   = clim_all_std[0:n_days - N_PREV]

    all_comp_mean = np.concatenate([all_prev_mean, all_cur_mean])
    all_comp_std  = np.concatenate([all_prev_std,  all_cur_std])

    # [FIX] low25 composite: Oct–Dec 用 clim_low25_full 的 Oct–Dec（即 Y-1），
    #                        Jan–Sep 用 clim_low25_full 的 Jan–Sep（即 Y）
    low25_prev_mean = clim_low25_mean[n_days - N_PREV:n_days]
    low25_prev_std  = clim_low25_std[n_days - N_PREV:n_days]
    low25_cur_mean  = clim_low25_mean[0:n_days - N_PREV]
    low25_cur_std   = clim_low25_std[0:n_days - N_PREV]

    low25_comp_mean = np.concatenate([low25_prev_mean, low25_cur_mean])
    low25_comp_std  = np.concatenate([low25_prev_std,  low25_cur_std])

    # === merged 年份 & special years ===
    years_merged = np.unique(var_merged.time.dt.year.values.astype(int))
    print(f"[BlockB_U] merged years ({level_label}):", years_merged)

    missing_special = YEARS_SPECIAL[~np.isin(YEARS_SPECIAL, years_merged)]
    if len(missing_special) > 0:
        print(
            f"⚠️ Warning: these special years are missing in merged ({level_label}):",
            missing_special,
        )

    # rcParams 样式
    rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    fig, ax = plt.subplots(1, 1, figsize=(18, 4.0), constrained_layout=True)

    x_full = np.arange(n_days)

    # special 年 daily 组合
    colors_spec = plt.cm.tab10(np.linspace(0, 1, len(YEARS_SPECIAL)))
    handles_years = []

    for i, y in enumerate(YEARS_SPECIAL):
        if y not in years_merged:
            continue

        cur = var_merged.sel(time=var_merged.time.dt.year == y)
        prev = var_merged.sel(time=var_merged.time.dt.year == (y - 1))

        if (cur.size < n_days) or (prev.size < n_days):
            print(
                f"⚠️ Year {y:04d} or {y-1:04d} does not have {n_days} days "
                f"({level_label}), skip."
            )
            continue

        cur_arr  = np.array(cur)
        prev_arr = np.array(prev)

        series_comp = np.concatenate(
            [
                prev_arr[n_days - N_PREV:n_days],
                cur_arr[0:n_days - N_PREV],
            ]
        )

        ax.plot(
            x_full,
            series_comp,
            linestyle="-",
            linewidth=1.5,
            color=colors_spec[i],
            label=f"Year {int(y):04d}",
        )
        handles_years.append(
            Line2D([0], [0], color=colors_spec[i], lw=1.5, label=f"Year {int(y):04d}")
        )

    # EXTR all-year 气候态 ±1σ
    ax.fill_between(
        x_full,
        all_comp_mean - all_comp_std,
        all_comp_mean + all_comp_std,
        color="0.85",
        alpha=0.9,
        linewidth=0,
        zorder=0,
    )
    ax.plot(
        x_full,
        all_comp_mean,
        linestyle="--",
        linewidth=1.8,
        color="black",
        label="EXTR climatology (all years)",
    )

    # EXTR low25 气候态 ±1σ
    ax.fill_between(
        x_full,
        low25_comp_mean - low25_comp_std,
        low25_comp_mean + low25_comp_std,
        facecolor="none",
        edgecolor="0.5",
        hatch="///",
        linewidth=0.0,
        zorder=1,
    )
    ax.plot(
        x_full,
        low25_comp_mean,
        linestyle="-",
        linewidth=2.0,
        color="black",
        label="EXTR low-ozone composite",
    )

    # 轴设置
    ax.set_xlim(0, n_days)
    ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(
        ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
         "May", "June", "July", "Aug", "Sep"]
    )
    ax.set_ylabel(f"U at 60°N, {level_label} (m s$^{{-1}}$)")

    patch_all = Patch(facecolor="0.85", edgecolor="none", label="EXTR all-year ±1σ")
    patch_low = Patch(facecolor="none", edgecolor="0.5", hatch="///",
                      label="EXTR low-ozone ±1σ")

    line_all = Line2D([0], [0], color="black", lw=1.8, ls="--",
                      label="EXTR climatology (all years)")
    line_low = Line2D([0], [0], color="black", lw=2.0, ls="-",
                      label="EXTR low-ozone composite")

    handles = handles_years + [line_all, patch_all, line_low, patch_low]
    ax.legend(handles=handles, loc="best", frameon=False, fontsize=8, ncol=2)

    ax.set_title(f"Zonal mean wind at 60°N, {level_label}")
    ax.grid(False)

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)

    print("[BlockB_U] Saved special-years vs climatology figure:")
    print("   ", out_png)
    print("   ", out_pdf)


def main_blockB_U():
    # ======== 读取缓存和 O3 极端年 ========
    u_extr_60 = xr.load_dataarray(U_EXTR_60N_NC)    # (time, lev[Pa])
    u_merged_60 = xr.load_dataarray(U_MERGED_60N_NC)

    lev_vals = u_extr_60["lev"].values  # Pa

    lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockB_U] Lowest 10 O3 years:", lowest10_years)
    print("[BlockB_U] Lowest 25% O3 years:", lowest25_years)

    # 目标层：10 hPa, 50 hPa
    target_levels_hpa = [1.0, 10.0, 50.0, 100.0]

    for target_hpa in target_levels_hpa:
        idx, lev_actual_hpa = get_level_index(lev_vals, target_hpa)
        level_label = f"{lev_actual_hpa:.1f} hPa"

        # ---- 取这一层的 EXTR & merged time series ----
        var_extr = u_extr_60.isel(lev=idx)
        var_merged = u_merged_60.isel(lev=idx)

        print(
            f"\n[BlockB_U] Processing level ~{target_hpa} hPa (index {idx}), "
            f"actual {lev_actual_hpa:.2f} hPa"
        )

        # 1) 极端 10 年 vs 非极端年
        out_png1 = os.path.join(
            U_ROOT, f"U_60N_{int(round(lev_actual_hpa))}hPa_evolution_min_10extr_200yrs.png"
        )
        out_pdf1 = os.path.join(
            U_ROOT, f"U_60N_{int(round(lev_actual_hpa))}hPa_evolution_min_10extr_200yrs.pdf"
        )
        plot_extremes_vs_others(
            var_extr=var_extr,
            lowest10_years=lowest10_years,
            level_label=level_label,
            out_png=out_png1,
            out_pdf=out_pdf1,
        )

        # 2) merged special years vs EXTR 气候态
        out_png2 = os.path.join(
            U_ROOT,
            f"U_60N_{int(round(lev_actual_hpa))}hPa_daily_special_years_vs_extr_climatology.png"
        )
        out_pdf2 = os.path.join(
            U_ROOT,
            f"U_60N_{int(round(lev_actual_hpa))}hPa_daily_special_years_vs_extr_climatology.pdf"
        )
        plot_special_vs_climatology(
            var_extr=var_extr,
            var_merged=var_merged,
            lowest25_years=lowest25_years,
            level_label=level_label,
            out_png=out_png2,
            out_pdf=out_pdf2,
        )

    print("[BlockB_U] Done.")


if __name__ == "__main__":
    main_blockB_U()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockC (U) — 60°N U 垂直时间剖面 anomaly + 显著性
#
# 输出一个 NetCDF：
#   U_vert_anom_special_60N.nc
#   coords:
#     ref_year (n_ref) : [8, 13, 14, 19]
#     lev             : 垂直层 (Pa, 与 merged 一致)
#     time_index      : 0..364 （拼接后的时间）
#     dayofyear       : 对应的 DOY 序列 [275..365, 1..274] (noleap)
#
#   variables (m/s):
#     anom_all        (ref_year, lev, time_index)
#     anom_low25      (ref_year, lev, time_index)
#     clim_all_comp   (lev, time_index)
#     clim_low25_comp (lev, time_index)
#
#   显著性掩膜（True = 显著）：
#     t_sig_all       (ref_year, lev, time_index)
#     b_sig_all       (ref_year, lev, time_index)
#     t_sig_low25     (ref_year, lev, time_index)
#     b_sig_low25     (ref_year, lev, time_index)
#
# 重要修正：
#   low-25% 气候态不再简单用极端年当年的日平均，
#   而是使用 “前一年 Oct–Dec + 当年 Jan–Sep” 的组合：
#     - Oct–Dec: lowest25_years-1 这些年的气候态
#     - Jan–Sep: lowest25_years   这些年的气候态
#   与 BlockB 中 U/O3 的时间拼接逻辑完全一致，消除 1.1–12.31 的假跳跃。
# ============================================================

import os
import numpy as np
import xarray as xr
from scipy.stats import t as t_dist

O3_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
U_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_U"
os.makedirs(U_ROOT, exist_ok=True)

# EXTR U 底层文件（已经是 200 年、zm）
EXTR_U_PATH = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_U.zm.nc"
)
# merged WACCM 文件
MERGED_FILE = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.merged.nc"
)

LOW25_TXT = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

OUT_NC = os.path.join(U_ROOT, "U_vert_anom_special_60N.nc")

N_PREV = 91
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)


def calc_u_60N_lev(ds, var="U"):
    """
    从 ds 中提取 U，计算 60°N 经向平均风场：
      - 若有 lon → 先经向平均
      - 选最近的 60°N 网格
      - 垂直维统一命名为 'lev'，并统一单位为 Pa
    返回 DataArray: (time, lev) [m/s]
    """
    da = ds[var]
    if "lon" in da.dims:
        da = da.mean(dim="lon")

    if "lat" not in da.dims:
        raise ValueError(f"{var} has no 'lat' dimension, dims={da.dims}")
    lat_vals = da["lat"].values
    idx_lat = int(np.argmin(np.abs(lat_vals - 60.0)))
    da_60 = da.isel(lat=idx_lat)

    lev_name = None
    for name in ("plev", "lev_p", "lev", "level"):
        if name in da_60.dims:
            lev_name = name
            break
    if lev_name is None:
        raise ValueError(
            f"{var}@60N has no vertical dim among ('plev','lev_p','lev','level'), "
            f"dims={da_60.dims}"
        )

    lev_vals = da_60[lev_name].values
    max_lev = float(np.nanmax(lev_vals))

    if max_lev <= 2000.0:
        lev_pa = lev_vals * 100.0
    else:
        lev_pa = lev_vals

    da_60 = da_60.rename({lev_name: "lev"})
    da_60 = da_60.assign_coords(lev=("lev", lev_pa))

    return da_60  # (time, lev)


def bootstrap_ci(data, nboot=5000, alpha=0.05, rng=None):
    data = np.asarray(data)
    data = data[~np.isnan(data)]
    n = data.size
    if n < 2:
        return np.nan, np.nan

    if rng is None:
        rng = np.random.default_rng()

    means = np.empty(nboot)
    for i in range(nboot):
        samp = rng.choice(data, size=n, replace=True)
        means[i] = np.mean(samp)

    low = np.percentile(means, 100 * alpha / 2.0)
    high = np.percentile(means, 100 * (1 - alpha / 2.0))
    return low, high


def compute_significance_for_baseline(
    base_anom,  # DataArray(time, lev) baseline anomalies
    anom_ref,   # np.array(lev, time_index) anomalies of special year
    doy_base,   # array(time,) DOY of baseline
    doy_comp,   # array(time_index,) DOY sequence of composite
    alpha=0.05,
    nboot=5000,
):
    """
    对每个 (lev, time_index)：
      - 使用 baseline 中该 DOY 的样本做 t 检验（均值=0）；
      - 使用 bootstrap CI 检查 obs 是否落在 CI 之外。
    返回:
      t_sig(lev, time_index), b_sig(lev, time_index)
    """
    base_vals = base_anom.values  # (time, lev)
    lev_n = base_vals.shape[1]
    t_len = anom_ref.shape[1]

    t_sig = np.zeros((lev_n, t_len), dtype=bool)
    b_sig = np.zeros((lev_n, t_len), dtype=bool)

    rng = np.random.default_rng(2025)

    for ti in range(t_len):
        day = int(doy_comp[ti])
        mask_day = (doy_base == day)
        if not np.any(mask_day):
            continue
        day_samples = base_vals[mask_day, :]  # (n_samples, lev)

        for li in range(lev_n):
            col = day_samples[:, li]
            col = col[~np.isnan(col)]
            if col.size < 2:
                continue

            obs = anom_ref[li, ti]
            se = np.std(col, ddof=1) / np.sqrt(col.size)
            if se == 0:
                continue
            tstat = obs / se
            pval = 2 * (1 - t_dist.cdf(abs(tstat), df=col.size - 1))
            t_sig[li, ti] = (pval < alpha)

            lo, hi = bootstrap_ci(col, nboot=nboot, alpha=alpha, rng=rng)
            if np.isnan(lo) or np.isnan(hi):
                continue
            b_sig[li, ti] = not (lo <= obs <= hi)

    return t_sig, b_sig


def main_blockC_U():
    # ===== 读取 O3 低25% 年份 =====
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockC_U] Lowest 25% O3 years:", lowest25_years)

    # ===== 1. 读取 EXTR U.zm，计算 60°N U(time,lev) =====
    print("[BlockC_U] Reading EXTR U.zm:", EXTR_U_PATH)
    ds_extr = xr.open_dataset(EXTR_U_PATH)
    u_extr_lev = calc_u_60N_lev(ds_extr, "U")  # (time, lev) m/s
    ds_extr.close()

    doy_extr = u_extr_lev.time.dt.dayofyear.values.astype(int)
    years_extr = u_extr_lev.time.dt.year.values.astype(int)
    print("[BlockC_U] EXTR years:", np.unique(years_extr))

    n_days = int(u_extr_lev.time.dt.dayofyear.max())
    print("[BlockC_U] Days per year (EXTR):", n_days)

    # ===== 2. EXTR 全部年份 / 低25% 年 日气候态 =====
    # 全部年份气候态（保持不动）
    clim_all = u_extr_lev.groupby("time.dayofyear").mean("time")  # (day, lev)

    # === [FIX] low-25% 气候态：Y-1 Oct–Dec + Y Jan–Sep ===
    #
    # base_low25_cur  : year ∈ lowest25_years 的当年（用于 Jan–Sep）
    # base_low25_prev : year ∈ lowest25_years-1 的前一年（用于 Oct–Dec）
    # 先分别对 dayofyear 做 climatology，再构造 clim_low25_full(dayofyear, lev)：
    #   DOY 1..(n_days-N_PREV)       → 使用 "当年" climatology
    #   DOY (n_days-N_PREV+1)..nDays → 使用 "前一年" climatology
    # 与 BlockB_U 中 U/O3 的时间拼接完全一致。
    # ------------------------------------------------------
    base_low25_cur = u_extr_lev.sel(time=u_extr_lev.time.dt.year.isin(lowest25_years))
    base_low25_prev = u_extr_lev.sel(
        time=u_extr_lev.time.dt.year.isin(lowest25_years - 1)
    )

    clim_low25_cur = base_low25_cur.groupby("time.dayofyear").mean("time")   # (day, lev)
    clim_low25_prev = base_low25_prev.groupby("time.dayofyear").mean("time") # (day, lev)

    # 从当年的 daily 气候态拷贝一份
    clim_low25_full = clim_low25_cur.copy()

    # Oct–Dec 的 DOY 范围（例如 noleap 时 275..365）
    oct_dec = np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int)

    # 用前一年的 climatology 替换 Oct–Dec 段
    clim_low25_full.loc[dict(dayofyear=oct_dec)] = (
        clim_low25_prev.sel(dayofyear=oct_dec).values
    )

    # 对应的 DOY 序列（1..365）
    doys = clim_all["dayofyear"].values.astype(int)

    # ===== 3. 读取 merged U，60°N =====
    print("[BlockC_U] Reading merged file:", MERGED_FILE)
    ds_merged = xr.open_dataset(MERGED_FILE)
    u_merged_lev = calc_u_60N_lev(ds_merged, "U")  # (time, lev) m/s
    ds_merged.close()

    years_merged = u_merged_lev.time.dt.year.values.astype(int)
    print("[BlockC_U] merged years:", np.unique(years_merged))

    lev = u_merged_lev["lev"].values
    lev_n = lev.size

    # ===== 4. 构造拼接的 DOY 序列：Y-1 Oct–Dec + Y Jan–Sep =====
    doy_comp = np.concatenate([
        np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int),
        np.arange(1, n_days - N_PREV + 1, dtype=int),
    ])
    t_len = doy_comp.size
    print("[BlockC_U] Composite DOY sequence:", doy_comp[:5], "...", doy_comp[-5:])

    # ===== 5. 预先计算 EXTR anomalies =====
    # all-years baseline
    clim_all_sel_for_extr = clim_all.sel(dayofyear=u_extr_lev.time.dt.dayofyear)
    base_anom_all = u_extr_lev - clim_all_sel_for_extr  # (time, lev)

    # [FIX] low-25% baseline 使用 clim_low25_full
    clim_low25_sel_for_extr = clim_low25_full.sel(
        dayofyear=u_extr_lev.time.dt.dayofyear
    )
    base_anom_low25 = u_extr_lev - clim_low25_sel_for_extr  # (time, lev)

    # ===== 6. 拼接基准气候态剖面（与 DOY_comp 对齐） =====
    clim_all_comp = clim_all.sel(dayofyear=doy_comp)           # (time_index, lev)
    # [FIX] 这里也必须用 clim_low25_full，确保跨年定义一致
    clim_low25_comp = clim_low25_full.sel(dayofyear=doy_comp)  # (time_index, lev)

    clim_all_comp_vals = clim_all_comp.transpose("lev", "dayofyear").values
    clim_low25_comp_vals = clim_low25_comp.transpose("lev", "dayofyear").values

    # ===== 7. 对每个 special year 计算 anomaly + 显著性 =====
    n_ref = len(YEARS_SPECIAL)

    anom_all_arr = np.zeros((n_ref, lev_n, t_len))
    anom_low_arr = np.zeros((n_ref, lev_n, t_len))

    t_sig_all = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_all = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    t_sig_low = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_low = np.zeros((n_ref, lev_n, t_len), dtype=bool)

    for yi, y in enumerate(YEARS_SPECIAL):
        if y not in years_merged:
            print(f"[BlockC_U] ⚠️ Year {y:04d} not found in merged, skip.")
            continue

        ref_cur = u_merged_lev.sel(time=u_merged_lev.time.dt.year == y)
        ref_prev = u_merged_lev.sel(time=u_merged_lev.time.dt.year == (y - 1))

        if ref_cur.time.size < n_days or ref_prev.time.size < n_days:
            print(f"[BlockC_U] ⚠️ Year {y:04d} or {y-1:04d} does not have {n_days} days, skip.")
            continue

        ref_cur_vals = np.array(ref_cur.transpose("time", "lev").values)    # (365, lev_n)
        ref_prev_vals = np.array(ref_prev.transpose("time", "lev").values)  # (365, lev_n)

        # 前一年最后 N_PREV 天 + 当年最前 n_days-N_PREV 天
        ref_comp_vals = np.concatenate([
            ref_prev_vals[n_days - N_PREV:n_days, :],
            ref_cur_vals[0:n_days - N_PREV, :],
        ], axis=0)  # (t_len, lev_n)

        ref_comp = ref_comp_vals.T  # (lev_n, t_len)

        # anomaly（m/s）
        anom_all = ref_comp - clim_all_comp_vals
        anom_low = ref_comp - clim_low25_comp_vals

        anom_all_arr[yi, :, :] = anom_all
        anom_low_arr[yi, :, :] = anom_low

        print(f"[BlockC_U] Computing significance for year {y:04d} vs ALL baseline ...")
        t_all, b_all = compute_significance_for_baseline(
            base_anom_all,
            anom_all,
            doy_extr,
            doy_comp,
            alpha=0.05,
            nboot=5000,
        )
        print(f"[BlockC_U] Computing significance for year {y:04d} vs LOW25 baseline ...")
        t_low, b_low = compute_significance_for_baseline(
            base_anom_low25,
            anom_low,
            doy_extr,
            doy_comp,
            alpha=0.05,
            nboot=5000,
        )

        t_sig_all[yi, :, :] = t_all
        b_sig_all[yi, :, :] = b_all
        t_sig_low[yi, :, :] = t_low
        b_sig_low[yi, :, :] = b_low

    # ===== 8. 保存到 NetCDF =====
    ds_out = xr.Dataset(
        coords={
            "ref_year": ("ref_year", YEARS_SPECIAL),
            "lev": ("lev", lev),
            "time_index": ("time_index", np.arange(t_len)),
            "dayofyear": ("time_index", doy_comp),
        },
        data_vars={
            "anom_all": (("ref_year", "lev", "time_index"), anom_all_arr),
            "anom_low25": (("ref_year", "lev", "time_index"), anom_low_arr),
            "clim_all_comp": (("lev", "time_index"), clim_all_comp_vals),
            "clim_low25_comp": (("lev", "time_index"), clim_low25_comp_vals),
            "t_sig_all": (("ref_year", "lev", "time_index"), t_sig_all),
            "b_sig_all": (("ref_year", "lev", "time_index"), b_sig_all),
            "t_sig_low25": (("ref_year", "lev", "time_index"), t_sig_low),
            "b_sig_low25": (("ref_year", "lev", "time_index"), b_sig_low),
        },
    )

    ds_out["anom_all"].attrs["units"] = "m s-1"
    ds_out["anom_low25"].attrs["units"] = "m s-1"
    ds_out["clim_all_comp"].attrs["units"] = "m s-1"
    ds_out["clim_low25_comp"].attrs["units"] = "m s-1"
    ds_out["lev"].attrs["units"] = "Pa"

    ds_out.attrs["description"] = (
        "Zonal mean zonal wind (U) at 60N, time–height anomalies (m s-1) "
        "for special years relative to EXTR all-years and low-25% climatology. "
        "Low-25% climatology is constructed using previous-year Oct–Dec and "
        "current-year Jan–Sep for the lowest-25% years. "
        "Significance masks (t-test & bootstrap) included."
    )

    ds_out.to_netcdf(OUT_NC)
    print(f"[BlockC_U] Saved vertical anomaly dataset to: {OUT_NC}")
    print("[BlockC_U] Done.")


if __name__ == "__main__":
    main_blockC_U()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockD (U) — 绘制 60°N U 时间×垂直剖面 anomaly 图
#
# 使用 BlockC_U 的输出 U_vert_anom_special_60N.nc：
#
#   对每个 ref_year（8, 13, 14, 19）：
#     1) baseline = EXTR all-years：
#         - t 检验 “不显著区域” hatch 图
#         - bootstrap “不显著区域” hatch 图
#     2) baseline = EXTR low-25%：
#         - t 检验 “不显著区域” hatch 图
#         - bootstrap “不显著区域” hatch 图
#
#   横坐标：0..364，与 BlockB 一致，刻度 Oct–Sep
#   纵坐标：pressure (log scale)，单位自动识别 Pa/hPa
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

U_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_U"
os.makedirs(U_ROOT, exist_ok=True)

IN_NC = os.path.join(U_ROOT, "U_vert_anom_special_60N.nc")

XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
                "May", "June", "July", "Aug", "Sep"]

# 全局样式
mpl.rc("xtick", direction="out", labelsize=8)
mpl.rc("ytick", direction="out", labelsize=8)
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8
mpl.rc("font", family="sans-serif")
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]
mpl.rc("axes", titlesize=11, labelsize=9, linewidth=0.8)
mpl.rc("legend", fontsize=7, frameon=False)
mpl.rc("figure", figsize=(6, 4), dpi=300)
mpl.rc("savefig", bbox="tight", pad_inches=0.1)


def guess_p_2_pa(lev_vals):
    """
    根据 lev 数值猜测单位：
      - 若 max(lev) <= 2000 → 认为是 hPa，乘 100 得 Pa
      - 否则认为是 Pa
    返回 (p_pa, unit_label)，其中 unit_label 表示原始单位（用于 y-label）
    """
    lev_vals = np.asarray(lev_vals)
    maxv = np.nanmax(lev_vals)
    if maxv <= 2000.0:
        return lev_vals * 100.0, "hPa"
    else:
        return lev_vals, "Pa"


def plot_time_height_anom(
    x_idx,
    lev_vals,
    anom_vals,
    clim_vals,
    nonsig_mask,
    ref_year,
    baseline_label,
    test_label,
    outfile,
):
    """
    绘制一张 U anomaly 时间×高度图：
      - 填色：anom_vals (m/s)
      - 黑实线：baseline climatology 的等值线 (m/s)
      - /// hatch：non-significant 区域
    """
    fig, ax = plt.subplots()

    x = np.asarray(x_idx)

    p_pa, p_unit = guess_p_2_pa(lev_vals)

    # 色阶范围（m/s）
    vlim = np.nanmax(np.abs(anom_vals))
    if np.isfinite(vlim) and vlim > 0:
        vmax = min(vlim, 30.0)  # 不要太夸张
    else:
        vmax = 10.0
    levels = np.linspace(-vmax, vmax, 31)

    cf = ax.contourf(
        x, p_pa, anom_vals,
        levels=levels,
        cmap="RdBu_r",
        extend="both",
        antialiased=True,
        alpha=0.85,
    )

    # baseline climatology 等值线
    clim_mean = np.nanmean(clim_vals)
    clim_std = np.nanstd(clim_vals)
    if np.isfinite(clim_std) and clim_std > 0:
        clim_levels = np.linspace(clim_mean - 1.5 * clim_std,
                                  clim_mean + 1.5 * clim_std, 7)
    else:
        clim_levels = 7

    CS = ax.contour(
        x, p_pa, clim_vals,
        levels=clim_levels,
        colors="k",
        linewidths=0.7,
    )
    ax.clabel(CS, inline=True, fontsize=6, fmt="%.1f")

    # 不显著区域 hatch
    if nonsig_mask is not None:
        mask_int = nonsig_mask.astype(int)
        ax.contourf(
            x, p_pa, mask_int,
            levels=[0.5, 1.5],
            colors="none",
            hatches=["///"],
            alpha=0,
        )
        patch = Patch(
            facecolor="white",
            hatch="///",
            edgecolor="black",
            label="Not significant (p ≥ 0.05)",
        )
        ax.legend(handles=[patch], loc="upper right")

    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_ylabel(f"Pressure ({p_unit})")

    ax.set_xlim(x[0], x[-1])
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICK_LABELS)

    ax.grid(True, which="major", linestyle="--", linewidth=0.4, alpha=0.5)

    ax.set_title(
        f"U anomaly (m s$^{{-1}}$), 60°N, Year {int(ref_year):04d}\n"
        f"Baseline: {baseline_label}, Mask: non-significant by {test_label}"
    )

    cbar = plt.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label("U anomaly (m s$^{-1}$)")

    plt.savefig(outfile, dpi=300)
    plt.close(fig)
    print("  Saved:", outfile)


def main_blockD_U():
    ds = xr.open_dataset(IN_NC)

    ref_years = ds["ref_year"].values
    lev = ds["lev"].values
    time_index = ds["time_index"].values

    anom_all = ds["anom_all"].values        # (ref, lev, time)
    anom_low = ds["anom_low25"].values
    clim_all = ds["clim_all_comp"].values   # (lev, time)
    clim_low = ds["clim_low25_comp"].values

    t_sig_all = ds["t_sig_all"].values
    b_sig_all = ds["b_sig_all"].values
    t_sig_low = ds["t_sig_low25"].values
    b_sig_low = ds["b_sig_low25"].values

    ds.close()

    for yi, y in enumerate(ref_years):
        # baseline = all-years, t-test
        nonsig_t_all = ~t_sig_all[yi, :, :]
        outfile = os.path.join(
            U_ROOT,
            f"U_anom_60N_year{int(y):04d}_vsALL_t_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_all[yi, :, :],
            clim_all,
            nonsig_t_all,
            ref_year=y,
            baseline_label="EXTR all-years climatology",
            test_label="t-test",
            outfile=outfile,
        )

        # baseline = all-years, bootstrap
        nonsig_b_all = ~b_sig_all[yi, :, :]
        outfile = os.path.join(
            U_ROOT,
            f"U_anom_60N_year{int(y):04d}_vsALL_bootstrap_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_all[yi, :, :],
            clim_all,
            nonsig_b_all,
            ref_year=y,
            baseline_label="EXTR all-years climatology",
            test_label="bootstrap",
            outfile=outfile,
        )

        # baseline = low25, t-test
        nonsig_t_low = ~t_sig_low[yi, :, :]
        outfile = os.path.join(
            U_ROOT,
            f"U_anom_60N_year{int(y):04d}_vsLOW25_t_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_low[yi, :, :],
            clim_low,
            nonsig_t_low,
            ref_year=y,
            baseline_label="EXTR low-25% climatology",
            test_label="t-test",
            outfile=outfile,
        )

        # baseline = low25, bootstrap
        nonsig_b_low = ~b_sig_low[yi, :, :]
        outfile = os.path.join(
            U_ROOT,
            f"U_anom_60N_year{int(y):04d}_vsLOW25_bootstrap_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_low[yi, :, :],
            clim_low,
            nonsig_b_low,
            ref_year=y,
            baseline_label="EXTR low-25% climatology",
            test_label="bootstrap",
            outfile=outfile,
        )

    print("[BlockD_U] All figures generated.")


if __name__ == "__main__":
    main_blockD_U()
